# 异常处理与调试

学习目标：能精准处理和传播异常，保留失败原因，并用回溯与调试器定位问题。

前置知识：条件与循环、函数调用与返回值、列表与字典、类的继承、标准库导入。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

调试命令已预置，可随章节顺序执行。

## 1 异常对象与层次

### 1.1 从失败的整数转换开始

异常（exception）使当前正常执行路径中断。try 放置可能失败的操作，except ValueError 只处理相应类型的异常；发生异常后，try 中剩余语句被跳过，不会从失败语句后自动续跑。

except ValueError as exc 中，exc 是捕获到的异常实例。type(exc) 查看类型，str(exc) 查看消息，exc.args 保存构造参数元组。异常类型说明失败类别，消息补充具体原因。

In [1]:
try:
    minutes = int("半小时")
    print("转换完成")  # 本例不会到达这里。
except ValueError as exc:
    print(type(exc).__name__)  # ValueError：字符串无法按整数解析。
    print(str(exc))  # 消息包含原始输入“半小时”。
    print(exc.args)  # 单元素元组，保存本次异常的消息。

print("继续处理下一项")  # 捕获完成后，从整个 try 语句之后继续。

ValueError
invalid literal for int() with base 10: '半小时'
("invalid literal for int() with base 10: '半小时'",)
继续处理下一项


### 1.2 基类、常见子类与捕获范围

BaseException 是异常层次的根；Exception 是常规异常的共同基类。KeyboardInterrupt、SystemExit 和 GeneratorExit 直接继承 BaseException，不属于 Exception，分别与用户中断、解释器退出、生成器或协程关闭有关；生成器机制在第 13 章展开。

因此，except Exception 不会捕获这些退出或中断信号；它也不意味着捕获到的错误一定能恢复。裸 except 和 except BaseException 范围更大，常规业务处理应选择能处理的具体类型。

| 原文名称 | 中文名称／含义 | 直接基类 |
| --- | --- | --- |
| ArithmeticError | 算术异常的基类 | Exception |
| ZeroDivisionError | 除法或取模的除数为零 | ArithmeticError |
| TypeError | 对象类型不支持相应操作 | Exception |
| ValueError | 类型合适，但值不合适且无更精确异常 | Exception |
| LookupError | 索引或键查找异常的基类 | Exception |
| IndexError | 序列下标越界 | LookupError |
| KeyError | 映射中不存在所需键 | LookupError |
| NameError | 找不到使用的局部或全局名称 | Exception |
| AttributeError | 属性引用或赋值失败 | Exception |
| OSError | 操作系统相关失败，含输入输出失败 | Exception |
| FileNotFoundError | 请求的文件或目录不存在 | OSError |
| ImportError | 模块加载或 from 导入失败 | Exception |
| ModuleNotFoundError | 无法定位导入模块等情况 | ImportError |
| RuntimeError | 不属于其他具体类别的运行错误 | Exception |
| AssertionError | assert 条件失败 | Exception |
| SyntaxError | 解析代码时发现语法错误 | Exception |

SyntaxError 也是异常类，但当前单元自身有语法错误时，还未进入其中的 try；不能靠同一段有语法错误的代码捕获它。后面仅使用语法正确的可运行反例。

In [2]:
print(issubclass(ZeroDivisionError, ArithmeticError))  # True。
print(issubclass(KeyError, LookupError))  # True。
print(issubclass(Exception, BaseException))  # True。

for exception_type in (KeyboardInterrupt, SystemExit, GeneratorExit):
    print(exception_type.__name__, issubclass(exception_type, Exception))
# 三行均为 False；这里只检查继承关系，不触发中断、退出或关闭。

True
True
True
KeyboardInterrupt False
SystemExit False
GeneratorExit False


## 2 捕获范围与执行路径

### 2.1 try、except、else、finally

except 处理匹配异常；else 只在 try 正常走到末尾、没有异常，也没有执行 return、break 或 continue 时运行。finally 在控制流离开这条 try 语句前运行，适合安排收尾动作；处理器或 else 失败时也会先经过它。

try 应尽量只包住预期会失败的语句。把后续成功操作放入 else，可以避免它们的异常被误当成前面操作的失败。

下例中的 finally 只显示结束标记；第 5 节再观察清理与控制流的边界。

In [3]:
for text in ("30", "半小时"):
    print("输入：", text)
    try:
        minutes = int(text)
    except ValueError:
        print("请输入整数字符串")
    else:
        print("分钟：", minutes)
    finally:
        print("本项结束")
# 第一项进入 else，第二项进入 except；两项最后都打印“本项结束”。

输入： 30
分钟： 30
本项结束
输入： 半小时
请输入整数字符串
本项结束


### 2.2 先捕获具体子类，再捕获基类

普通 except 从上到下选择第一个匹配分支，一次最多执行一个。基类可以匹配其子类异常，因此把基类写在前面会遮住后面的子类分支。

多个异常需要相同处理时，可在一个括号元组中列出类型。下面用下标读取演示：KeyError 和 IndexError 都是 LookupError 的子类；最后一个分支处理剩余的 LookupError。

In [4]:
for container, key in [({}, "title"), ([], 0)]:
    try:
        container[key]
    except (KeyError, IndexError) as exc:
        print("缺少条目：", type(exc).__name__)
    except LookupError:
        print("其他查找失败")
# 依次打印 KeyError、IndexError；若把 LookupError 分支放在前面，
# 两种异常都会被它先匹配，具体类型分支不会执行。

缺少条目： KeyError
缺少条目： IndexError


### 2.3 沿调用栈传播

try 也能捕获其间调用的函数内部产生的异常。当前函数没有匹配处理器时，异常沿调用关系向外传播，直至找到处理器；没有处理器时成为未处理异常。

函数无力恢复时可以直接传播，无需每层都包一遍 try。传播会退出未完成的调用，不会执行失败调用之后的普通语句。

In [5]:
def parse_minutes(text):
    """把分钟文字转换为整数，让转换失败自然传播。"""
    return int(text)


def total_minutes(text):
    """读取一项时长并加上固定休息时间。"""
    minutes = parse_minutes(text)
    return minutes + 5


try:
    print(total_minutes("半小时"))
except ValueError:
    print("总时长计算失败：分钟文字不是整数")
# parse_minutes 失败后，total_minutes 的加法和外层 print 都未执行。

总时长计算失败：分钟文字不是整数


### 2.4 处理器的边界与异常变量的寿命

同一条 try 的 except 只捕获 try 主体产生的异常，不能捕获同级 else 或另一个 except 内的新异常；这些异常应向外传播。

as 绑定的异常变量在 except 结束时会被清除，以避免异常、回溯与局部变量长期相互引用。需要事后使用诊断信息时，可以先保存消息；不要把该名称当作普通的持久变量。

In [6]:
try:
    try:
        minutes = int("30")
    except ValueError:
        print("转换失败")  # 转换成功，本分支不执行。
    else:
        raise ValueError("后续检查：时长超过本次上限")
except ValueError as boundary_error:
    message = str(boundary_error)

print(message)  # 后续检查的异常到达外层，未进入“转换失败”分支。
try:
    print(boundary_error)
except NameError:
    print("异常变量已清除，消息仍可使用")
# NameError 是本例特意观察的边界；不要在实际代码中依赖已清除名称。

后续检查：时长超过本次上限
异常变量已清除，消息仍可使用


## 3 主动引发、重新引发与自定义异常

### 3.1 用 raise 拒绝不合法输入

raise 后可写异常实例，或可无参实例化的异常类；例如 raise ValueError 等价于引发一个无消息的 ValueError 实例。实际校验通常提供包含失败对象与条件的消息。

参数类型不合适时用 TypeError；类型合适但值越界时用 ValueError。异常对象必须属于 BaseException 层次，不能直接 raise 字符串。下面的接口只接受普通整数，不接受 bool。

In [7]:
def require_minutes(value):
    """接受非负普通整数分钟，拒绝其他类型和负数。"""
    if type(value) is not int:
        raise TypeError("分钟数必须是普通整数")
    if value < 0:
        raise ValueError(f"分钟数不能为负数：{value}")
    return value


for value in (0, 30, -1, "30", True):
    try:
        print("接收：", require_minutes(value))
    except (TypeError, ValueError) as exc:
        print(type(exc).__name__, str(exc))
# 0 和 30 成功；-1 为 ValueError；字符串与 bool 为 TypeError。

接收： 0
接收： 30
ValueError 分钟数不能为负数：-1
TypeError 分钟数必须是普通整数
TypeError 分钟数必须是普通整数


### 3.2 裸 raise 保留当前异常

不带表达式的 raise 重新引发当前正在处理的异常，不需要重新构造实例。可以先用 add_note 添加字符串补充说明，再让外层继续处理；备注也会进入标准回溯显示。

没有活动异常时执行裸 raise 会引发 RuntimeError。它与“创建一个新异常”用途不同；补充说明后重新抛出，也不等于恢复成功。

In [8]:
original_error = ValueError("时长字段格式不正确")
try:
    try:
        raise original_error
    except ValueError as exc:
        exc.add_note("记录编号：L11；字段：minutes")
        raise
except ValueError as exc:
    print(exc is original_error)  # True：重新引发的是同一个异常实例。
    print(exc.__notes__)  # 保存刚补充的记录与字段信息。

try:
    raise
except RuntimeError as exc:
    print(type(exc).__name__)  # RuntimeError：这里已没有活动异常。

True
['记录编号：L11；字段：minutes']
RuntimeError


### 3.3 用领域异常表达接口失败

自定义异常通常直接或间接继承 Exception，并以 Error 结尾。它仍是普通类，可以保存调用者需要的字段；这里的异常继承是第 10 章继承机制的一种应用。

选择一个异常基类即可，不要为了同时匹配多个类别而多重继承内置异常：构造参数和内部布局可能冲突。下面用 InvalidMinutesError 表达时长规则失败，并保留对应记录编号。

In [9]:
class InvalidMinutesError(Exception):
    """表示某条学习记录的时长不符合接口要求。"""

    def __init__(self, record_id, message):
        """保存记录编号，并组成可读的异常消息。"""
        self.record_id = record_id
        super().__init__(f"记录 {record_id}：{message}")


try:
    raise InvalidMinutesError("L11", "分钟数不能为负数")
except InvalidMinutesError as exc:
    print(exc.record_id)  # L11：处理器可直接读取结构化信息。
    print(str(exc))  # 记录 L11：分钟数不能为负数。
# 后续异常转换继续使用这个类，不依赖其他 Notebook 的定义。

L11
记录 L11：分钟数不能为负数


## 4 异常链保留失败原因

### 4.1 隐式异常上下文

处理一个异常时又引发新异常，Python 会把旧异常记录在新异常的 \_\_context\_\_ 中。这是隐式异常链（implicit exception chaining），说明“处理前一个异常期间又发生了异常”，不自动断言两者是业务上的因果关系。

下面故意在捕获 KeyError 后引发 RuntimeError，观察新异常保存的上下文。

In [10]:
try:
    try:
        {}["minutes"]
    except KeyError:
        raise RuntimeError("记录读取阶段失败")
except RuntimeError as exc:
    print(type(exc.__context__).__name__)  # KeyError：原异常仍可追踪。
    print(exc.__cause__)  # None：未显式指定直接原因。
    print(exc.__suppress_context__)  # False：默认显示隐式上下文。

KeyError
None
False


### 4.2 用 from 明确直接原因

raise new_error from cause 中，new_error 是要向外提供的异常，cause 是其直接原因，二者可为异常实例或可无参构造的异常类。显式异常链把原因保存在 \_\_cause\_\_ 中，并使默认回溯优先显示这条原因链。

转换底层错误为领域错误时，from 能保留原始问题。本例复用 InvalidMinutesError，只转换确定的整数解析失败；其他错误仍自然传播。

In [11]:
def read_minutes(record_id, text):
    """读取记录时长，并把整数解析失败转换为领域异常。"""
    try:
        return int(text)
    except ValueError as exc:
        raise InvalidMinutesError(record_id, "时长必须写成整数") from exc


try:
    read_minutes("L11", "半小时")
except InvalidMinutesError as exc:
    print(str(exc))  # 对调用者说明哪条记录、哪个规则失败。
    print(type(exc.__cause__).__name__)  # ValueError：直接原因。
    print(exc.__cause__ is exc.__context__)  # True：此例中二者引用相同。
    print(exc.__suppress_context__)  # True：按显式原因链显示。

记录 L11：时长必须写成整数
ValueError
True
True


### 4.3 from None 抑制显示，不删除上下文

raise new_error from None 会把 \_\_cause\_\_ 设为 None，并将 \_\_suppress\_context\_\_ 设为 True；默认回溯因此只显示新异常，旧异常仍保存在 \_\_context\_\_ 中。

这适合接口有意省略底层实现细节的情况，不应误认为原异常从内存中消失。本例将缺少字典键转换成“缺少时长字段”的领域错误。

In [12]:
try:
    try:
        {}["minutes"]
    except KeyError:
        raise InvalidMinutesError("L11", "缺少时长字段") from None
except InvalidMinutesError as exc:
    print(exc.__cause__, exc.__suppress_context__)  # None True。
    print(type(exc.__context__).__name__)  # KeyError：仍能读取上下文。
    hidden_context_error = exc
# 暂存该实例，供第 8 节比较实际回溯文本；它会保留关联回溯。

None True
KeyError


## 5 finally 与清理失败

### 5.1 返回之前也要执行清理

finally 不只在异常时执行；try 中准备 return 时也会先执行 finally。若 finally 正常结束，原返回或未处理异常继续生效。

下面用列表记录收尾次序，不创建文件或连接。这里只学习控制流；文件读写在第 12 章展开，上下文管理器如何组织资源清理在第 15 章展开。

In [13]:
def finish_task(should_fail, events):
    """记录任务开始与收尾，并在指定情况下引发失败。"""
    try:
        events.append("开始")
        if should_fail:
            raise ValueError("任务输入不合法")
        return "完成"
    finally:
        events.append("收尾")


for should_fail in (False, True):
    events = []
    try:
        print(finish_task(should_fail, events))
    except ValueError as exc:
        print(str(exc))
    print(events)  # 两次均为 ['开始', '收尾']；失败仍传播到调用方。

完成
['开始', '收尾']
任务输入不合法
['开始', '收尾']


### 5.2 清理又失败时，哪个异常向外传播

若原异常尚未处理，finally 又引发新异常，新异常会向外传播，旧异常成为它的隐式上下文。只看最后一行消息可能漏掉最早的失败，应结合异常链检查。

下面的两个 raise 明确代表“任务失败”和“清理失败”，用于观察传播规则。实际清理应针对具体操作设计；捕获清理异常后返回成功值会隐瞒失败。

In [14]:
try:
    try:
        raise ValueError("任务：无法解析时长")
    finally:
        raise RuntimeError("清理：释放资源失败")
except RuntimeError as exc:
    print(str(exc))  # 向外传播的是“清理：释放资源失败”。
    print(type(exc.__context__).__name__)  # ValueError。
    print(str(exc.__context__))  # 原任务失败信息仍在异常链中。

清理：释放资源失败
ValueError
任务：无法解析时长


### 5.3 finally 中的跳转会覆盖原控制流

Python 3.12 中，finally 执行 return、break 或 continue，会丢弃尚待重新引发的异常；finally 的 return 还会覆盖 try 中已经计算好的返回值。

因此，清理块应专注收尾，避免用这些语句离开 finally。下例是用于辨认风险的反例，不能作为“容错成功”的写法。

In [15]:
def unsafe_return(should_fail):
    """反例：finally 的 return 覆盖原返回值或原异常。"""
    try:
        if should_fail:
            raise ValueError("真实任务失败")
        return "真实结果"
    finally:
        return "覆盖后的结果"


print(unsafe_return(False))  # 覆盖后的结果：原返回值丢失。
print(unsafe_return(True))  # 同样返回，ValueError 没有传播。

for number in (1, 2):
    try:
        raise ValueError("本应传播")
    finally:
        break
print("break 后：", number)  # 1；异常被丢弃，循环提前退出。

visited = []
for number in (1, 2):
    try:
        raise ValueError("本应传播")
    finally:
        visited.append(number)
        continue
print("continue 后：", visited)  # [1, 2]；两个异常均被丢弃。

覆盖后的结果
覆盖后的结果
break 后： 1
continue 后： [1, 2]


## 6 异常组与 except\*

### 6.1 把多个失败作为一个异常传播

ExceptionGroup 将多个异常实例包装成一个可引发的异常；它仍是异常对象，内部保存多个成员异常。异常链表示前后关联的失败，异常组表示同时报告的一批失败，两者可以并存。

ExceptionGroup 的第一个参数是组消息，第二个参数是非空异常序列，成员必须是 Exception 的实例，可以包含嵌套 ExceptionGroup。exceptions 属性以元组保存成员。

普通 except 匹配异常组对象本身，不会自动深入匹配组内的 ValueError。下面只收集本批预期的转换失败；输入很小，继续检查其余项有明确意义。

In [16]:
conversion_errors = []
for text in ("10", "半小时", "稍后"):
    try:
        int(text)
    except ValueError as exc:
        exc.add_note(f"输入项：{text}")
        conversion_errors.append(exc)

try:
    if conversion_errors:
        raise ExceptionGroup("时长批量转换失败", conversion_errors)
except ValueError:
    print("不会匹配组内成员")  # 抛出对象的类型是 ExceptionGroup。
except ExceptionGroup as group:
    print(group.message)  # 时长批量转换失败。
    print(len(group.exceptions))  # 2；成功的 "10" 不在错误组中。
    print([type(item).__name__ for item in group.exceptions])
    # ['ValueError', 'ValueError']：普通 except 捕获整个组。

时长批量转换失败
2
['ValueError', 'ValueError']


### 6.2 按类型处理子组，让未处理部分继续传播

except\* 按类型从组中分出匹配部分，as 绑定的仍是异常组，原有嵌套结构会保留。同一条 try 可执行多个 except\* 分支；每个分支至多一次，每个成员只交给第一个匹配它的分支。

没有匹配的成员在最后重新引发；处理器自身引发的新异常也会在最后向外传播，必要时与剩余异常组合。新异常不会交给同级后续分支再次处理。

同一条 try 不能混用 except 与 except\*；下面用外层普通 except 接住剩余组，保证示例能继续执行。

In [17]:
try:
    try:
        raise ExceptionGroup(
            "批量检查",
            [
                ValueError("时长越界"),
                ExceptionGroup("子批次", [TypeError("类型不合适")]),
                KeyError("title"),
            ],
        )
    except* ValueError as matched:
        print("数值问题：", len(matched.exceptions))  # 1。
    except* TypeError as matched:
        nested = matched.exceptions[0]
        print(nested.message, type(nested.exceptions[0]).__name__)
        # 子批次 TypeError：嵌套结构仍在，没有扁平化。
except ExceptionGroup as remaining:
    print("向外传播：", type(remaining.exceptions[0]).__name__)
    # KeyError：两个处理器都没匹配它，因此保留在剩余组中。

数值问题：

 1
子批次 TypeError
向外传播： KeyError


### 6.3 异常组的类型边界

BaseExceptionGroup 可以包含任意 BaseException 实例；ExceptionGroup 只接收 Exception 实例。若给 BaseExceptionGroup 的所有成员都属于 Exception，构造时会自动返回 ExceptionGroup。

except\* 也能匹配单个非组异常，此时自动包装成消息为空的组。except\* 后不能写 ExceptionGroup 或 BaseExceptionGroup，也不能省略匹配类型；处理器本身不允许用 return、break 或 continue 跳出。

下面只构造包含中断信号的组来检查其类型，不实际引发中断。

In [18]:
ordinary = BaseExceptionGroup("普通失败", [ValueError("输入不合法")])
control = BaseExceptionGroup("包含中断", [KeyboardInterrupt()])
print(type(ordinary).__name__)  # ExceptionGroup：自动选择较具体类型。
print(type(control).__name__)  # BaseExceptionGroup。
print(isinstance(control, Exception))  # False。

try:
    ExceptionGroup("错误成员", [KeyboardInterrupt()])
except TypeError as exc:
    print(type(exc).__name__)  # TypeError：中断信号不属于 Exception。

try:
    raise ValueError("单项失败")
except* ValueError as matched:
    print(type(matched).__name__, repr(matched.message))
    # ExceptionGroup ''：单个异常也被包装成组。

ExceptionGroup
BaseExceptionGroup
False
TypeError
ExceptionGroup ''


## 7 assert 与输入校验

assert condition, message 中，condition 是需要成立的条件，message 是失败时的说明；消息可以省略。条件为假时引发 AssertionError，适合开发时检查内部假设。

Python 的 -O 优化模式会移除断言及条件求值。因此不能用 assert 替代必要输入校验，也不要把必需的函数调用或状态修改放在断言条件里。

下面假定排序逻辑应产生有序结果，用断言检查这个内部假设；外部输入则沿用第 3 节 require_minutes 的 if 与 raise，无论是否启用优化都必须校验。

In [19]:
ordered = sorted([30, 10, 20])
assert ordered == [10, 20, 30], "排序结果违反内部假设"
print(ordered)  # [10, 20, 30]：正常断言没有单独输出。

try:
    assert 10 <= 5, "反例：下界不能大于上界"
except AssertionError as exc:
    print(str(exc))  # 正常非优化执行时，显示这个内部假设失败。

try:
    require_minutes(-1)
except ValueError as exc:
    print(str(exc))  # 必需的输入校验通过 if/raise 保留，不依赖 assert。

[10, 20, 30]
反例：下界不能大于上界
分钟数不能为负数：-1


## 8 用 traceback 阅读失败位置

### 8.1 显示完整回溯与异常链

回溯（traceback）记录异常经过的调用位置。通常先看最后的异常类型与消息，再沿调用条目查找出错语句与参数来源；最后列出的调用帧最接近失败位置。

traceback.print_exception 接收异常实例，默认同时显示适用的异常链。下例复用 read_minutes，并将回溯输出到 sys.stdout，方便在本单元读取；add_note 补充的记录信息会显示在异常消息后。

In [20]:
import sys
import traceback

try:
    read_minutes("L11", "半小时")
except InvalidMinutesError as exc:
    exc.add_note("操作：汇总本周学习时长")
    traceback.print_exception(exc, file=sys.stdout)
# 先显示 ValueError 及其调用位置，再显示直接原因连接文字，
# 最后是 InvalidMinutesError、记录编号和附加说明。
# Notebook 回溯中的临时文件标识随内核运行变化，以函数名与代码行为准。

Traceback (most recent call last):
  File "C:\Users\ZHUANG\AppData\Local\Temp\nb11-final-olrefyre\ipykernel_4644\2351151693.py", line 4, in read_minutes
    return int(text)
           ^^^^^^^^^
ValueError: invalid literal for int() with base 10: '半小时'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\ZHUANG\AppData\Local\Temp\nb11-final-olrefyre\ipykernel_4644\1862417636.py", line 5, in <module>
    read_minutes("L11", "半小时")
  File "C:\Users\ZHUANG\AppData\Local\Temp\nb11-final-olrefyre\ipykernel_4644\2351151693.py", line 6, in read_minutes
    raise InvalidMinutesError(record_id, "时长必须写成整数") from exc
InvalidMinutesError: 记录 L11：时长必须写成整数
操作：汇总本周学习时长


### 8.2 提取位置与格式化诊断文本

异常的 \_\_traceback\_\_ 属性保存回溯对象。traceback.extract_tb 将它转成帧摘要序列；每个摘要的 name、lineno、filename 分别是函数名、行号与文件标识。它们描述实际调用位置，不是错误原因的自动判断。

traceback.format_exception 返回字符串列表，拼接后得到与 print_exception 相同的诊断文本。下面同时复用第 4 节保存的 hidden_context_error，观察 from None 对显示的影响；普通的 chain=False 则是本次格式化主动省略异常链，不会修改异常对象。

In [21]:
try:
    total_minutes("半小时")  # 复用第 2 节的两层调用。
except ValueError as exc:
    frames = traceback.extract_tb(exc.__traceback__)
    print([frame.name for frame in frames[-2:]])
    # ['total_minutes', 'parse_minutes']：从外层调用走向出错函数。
    print(all(frame.lineno > 0 for frame in frames))  # True：行号从 1 开始。
    diagnostic = "".join(traceback.format_exception(exc, chain=False))
    print("ValueError:" in diagnostic)  # True：仍保留本次异常和回溯。

visible = "".join(traceback.format_exception(hidden_context_error))
print(visible, end="")  # 只显示领域异常，不显示隐式 KeyError 的回溯。
del hidden_context_error  # 诊断结束后，不再额外持有这个异常及其回溯。

['total_minutes', 'parse_minutes']
True
True
Traceback (most recent call last):
  File "C:\Users\ZHUANG\AppData\Local\Temp\nb11-final-olrefyre\ipykernel_4644\963095364.py", line 5, in <module>
    raise InvalidMinutesError("L11", "缺少时长字段") from None
InvalidMinutesError: 记录 L11：缺少时长字段


## 9 用 pdb 观察执行过程

### 9.1 逐行执行并查看局部变量

pdb 是标准库调试器。pdb.Pdb.runcall 在进入指定函数时停住，并在函数结束后返回该函数的结果。停住时可以查看当前帧中的变量，再决定执行下一行还是进入被调用函数。

| 命令 | 中文名称／含义 |
| --- | --- |
| p expression | 求值并显示 expression；这里 expression 是当前帧可访问的表达式 |
| n | next，在当前函数走到下一行或返回，通常不进入被调函数 |
| s | step，走到下一个可停位置，可以进入被调函数 |
| l | list，列出当前附近的源码 |
| w | where，显示调用栈及当前帧 |
| u | up，选择上一层调用帧 |
| d | down，选择下一层调用帧 |
| b line_number | break，在行号 line_number 指定的位置设断点 |
| c | continue，继续运行，遇到断点再停 |
| q | quit，退出调试器并中止被调试程序 |

为自动展示会话，本例提前使用 io.StringIO：它是内存中的文本流，分别保存预置命令与调试输出，getvalue 取回输出，close 关闭流。文本流的完整用法在第 12 章展开。

当前内核的 ipykernel 将 pdb.Pdb 替换为间接继承标准库 Pdb 的 IPython.core.debugger.InterruptiblePdb，因此实际会话使用 IPython 调试器，提示符为 ipdb>，并带有配色。

这里实际执行 n、p、c，命令来自字符串而不是人手输入；readrc=False 避免读取个人调试配置，nosigint=True 避免改变中断信号处理器，use_rawinput=False 确保读取所给输入流。

In [22]:
import io
import pdb


def buggy_mean(values):
    """反例：计算平均值时错误地把样本数多加一。"""
    total = sum(values)
    count = len(values) + 1
    return total / count


commands = io.StringIO(
    "p values\nn\np total\nn\np count\np len(values)\nc\n"
)
transcript = io.StringIO()
try:
    debugger = pdb.Pdb(
        stdin=commands, stdout=transcript, readrc=False, nosigint=True
    )
    debugger.use_rawinput = False
    result = debugger.runcall(buggy_mean, [40, 80])
    print(transcript.getvalue(), end="")
    print("函数结果：", result)  # 40.0；正确平均值应为 60.0。
finally:
    commands.close()
    transcript.close()
# 会话先看到 values=[40, 80]；两次 n 后分别看到 total=120、count=3。
# p len(values) 显示 2，因此可以定位多加一的语句，而非猜测除法出错。

> c:\users\zhuang\appdata\local\temp\nb11-final-olrefyre\ipykernel_4644\671275168.py(7)buggy_mean()

ipdb> [40, 80]
ipdb> > c:\users\zhuang\appdata\local\temp\nb11-final-olrefyre\ipykernel_4644\671275168.py(8)buggy_mean()

ipdb> 120
ipdb> > c:\users\zhuang\appdata\local\temp\nb11-final-olrefyre\ipykernel_4644\671275168.py(9)buggy_mean()

ipdb> 3
ipdb> 2
ipdb> 

函数结果： 40.0


### 9.2 breakpoint 与手工调试的关系

内置 breakpoint() 调用 sys.breakpointhook；Python 的默认钩子进入 pdb.set_trace。默认实现还会读取 PYTHONBREAKPOINT，值为 "0" 时不进入调试器；宿主应用也可能替换钩子，Notebook 中的实际行为未必与普通终端相同。

下面暂时把钩子指向带有预置命令的 Pdb.set_trace，在修正函数的关键位置实际调用 breakpoint。本例在后续 return 行停住，尚未计算返回表达式；finally 恢复原钩子并关闭两个内存流。

手工调试时，可在独立终端脚本的目标位置保留 breakpoint()，到提示符后逐条输入 p、n、s、c 等命令。这里不需要手工输入，也不要从自动执行单元中删除命令流后直接运行交互断点。

In [23]:
# 沿用前面导入的 io、pdb、sys。
def checked_mean(values):
    """拒绝空输入，并在平均值计算前提供调试观察点。"""
    if not values:
        raise ValueError("平均值计算至少需要一个数值")
    total = sum(values)
    count = len(values)
    breakpoint()
    return total / count


commands = io.StringIO("p total\np count\np total / count\nc\n")
transcript = io.StringIO()
original_hook = sys.breakpointhook
try:
    debugger = pdb.Pdb(
        stdin=commands, stdout=transcript, readrc=False, nosigint=True
    )
    debugger.use_rawinput = False
    sys.breakpointhook = debugger.set_trace
    result = checked_mean([40, 80])
    print(transcript.getvalue(), end="")
    print("修正结果：", result)  # 60.0；会话中 total=120、count=2。
finally:
    sys.breakpointhook = original_hook
    commands.close()
    transcript.close()
# 调试结束后，正常业务代码应移除 checked_mean 中用于观察的断点。

> c:\users\zhuang\appdata\local\temp\nb11-final-olrefyre\ipykernel_4644\4223695931.py(9)checked_mean()

ipdb> 120
ipdb> 2
ipdb> 60.0
ipdb> 修正结果： 60.0


## 本章小结

（1）异常由类型与实例信息共同描述；捕获具体类型，把 try 限于可处理的操作，让无法处理的错误向外传播。

（2）else 表达成功后的路径，finally 表达离开前的收尾；清理失败会形成异常链，清理中的跳转可能覆盖原结果或异常。

（3）裸 raise 重新引发当前异常；领域异常与 from 保留接口信息和原始原因，from None 只影响默认显示。

（4）异常组汇总多个失败，except\* 分别处理匹配子组，未处理部分继续传播。

（5）assert 检查内部假设，输入校验使用 if 与 raise；traceback 提供位置和异常链，pdb 可以停住代码并查看当时的变量。

自查：能否区分“捕获并恢复”“转换后传播”“清理后继续传播”，并说明为什么只打印最后一个错误消息可能漏掉最早的失败？

## 练习

### 练习 1：预测执行次序

先不要运行，写出两次调用的返回值和 events 内容，指出 else 与 finally 分别是否执行。预测完成后运行核对，并说明在 try 中 return 与在 try 中走到末尾有什么区别。

In [24]:
def predict_flow(early, events):
    """练习：记录正常结束与提前返回经过的分支。"""
    try:
        events.append("try")
        if early:
            return "early"
    except ValueError:
        events.append("except")
    else:
        events.append("else")
    finally:
        events.append("finally")
    return "normal"


for early in (False, True):
    events = []
    print(predict_flow(early, events), events)

normal ['try', 'else', 'finally']
early ['try', 'finally']


### 练习 2：校验记录并保留原因

实现 parse_record_minutes(record_id, text)，复用本章 InvalidMinutesError。text 约定为字符串：转换为整数后，只接受 0 到 120（含端点）。整数解析失败时用 from 保留 ValueError；越界时直接引发领域异常。

检查 "0"、"120" 原样得到整数，"-1"、"121" 引发领域异常，"半小时" 的领域异常中 \_\_cause\_\_ 为 ValueError，并保留 record_id。只捕获需要转换的异常，不使用 assert 做输入校验。

In [25]:
# 在此实现 parse_record_minutes，并检查两个端点、两个越界值和解析失败。
# 捕获预期的 InvalidMinutesError 后检查 record_id 与 __cause__。
pass

### 练习 3：让一批失败分别到达合适的处理器

创建包含 ValueError("时长越界")、TypeError("时长类型不合适")、KeyError("title") 的 ExceptionGroup。内层使用 except\* 分别处理 ValueError 与 TypeError；外层普通 except 捕获未处理的剩余组。

核对两个内层处理器各运行一次，剩余组只包含 KeyError。再把 ValueError 分支改为引发 RuntimeError("校验处理器失败")，观察外层同时收到新的 RuntimeError 和未处理的 KeyError；TypeError 分支仍应执行。不要把 except 和 except\* 混写在同一条 try 中。

In [26]:
# 在此实现原始版本，再实现处理器自身失败的版本。
# 用成员类型与处理器执行记录核对要求；不需要额外库或并发任务。
pass

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Python 官方文档（3.12） | [教程：异常处理与传播](https://docs.python.org/3.12/tutorial/errors.html#handling-exceptions)、[自定义异常](https://docs.python.org/3.12/tutorial/errors.html#user-defined-exceptions)、[汇集多个异常](https://docs.python.org/3.12/tutorial/errors.html#raising-and-handling-multiple-unrelated-exceptions)；[异常层次](https://docs.python.org/3.12/library/exceptions.html#exception-hierarchy)、[BaseException 与 args](https://docs.python.org/3.12/library/exceptions.html#BaseException)、[Exception](https://docs.python.org/3.12/library/exceptions.html#Exception)、[KeyboardInterrupt](https://docs.python.org/3.12/library/exceptions.html#KeyboardInterrupt)、[SystemExit](https://docs.python.org/3.12/library/exceptions.html#SystemExit)、[GeneratorExit](https://docs.python.org/3.12/library/exceptions.html#GeneratorExit)、[ArithmeticError](https://docs.python.org/3.12/library/exceptions.html#ArithmeticError)、[ZeroDivisionError](https://docs.python.org/3.12/library/exceptions.html#ZeroDivisionError)、[TypeError](https://docs.python.org/3.12/library/exceptions.html#TypeError)、[ValueError](https://docs.python.org/3.12/library/exceptions.html#ValueError)、[LookupError](https://docs.python.org/3.12/library/exceptions.html#LookupError)、[IndexError](https://docs.python.org/3.12/library/exceptions.html#IndexError)、[KeyError](https://docs.python.org/3.12/library/exceptions.html#KeyError)、[NameError](https://docs.python.org/3.12/library/exceptions.html#NameError)、[AttributeError](https://docs.python.org/3.12/library/exceptions.html#AttributeError)、[OSError](https://docs.python.org/3.12/library/exceptions.html#OSError)、[FileNotFoundError](https://docs.python.org/3.12/library/exceptions.html#FileNotFoundError)、[ImportError](https://docs.python.org/3.12/library/exceptions.html#ImportError)、[ModuleNotFoundError](https://docs.python.org/3.12/library/exceptions.html#ModuleNotFoundError)、[RuntimeError](https://docs.python.org/3.12/library/exceptions.html#RuntimeError)、[AssertionError](https://docs.python.org/3.12/library/exceptions.html#AssertionError)、[SyntaxError](https://docs.python.org/3.12/library/exceptions.html#SyntaxError)；[except 的顺序、传播与变量清除](https://docs.python.org/3.12/reference/compound_stmts.html#except-clause)、[else 的进入条件](https://docs.python.org/3.12/reference/compound_stmts.html#else-clause)、[finally 与控制流覆盖](https://docs.python.org/3.12/reference/compound_stmts.html#finally-clause)；[raise、裸 raise 与原因链](https://docs.python.org/3.12/reference/simple_stmts.html#the-raise-statement)、[异常继承限制](https://docs.python.org/3.12/library/exceptions.html#inheriting-from-built-in-exceptions)、[异常上下文与 from None](https://docs.python.org/3.12/library/exceptions.html#exception-context)、[add_note](https://docs.python.org/3.12/library/exceptions.html#BaseException.add_note)；[ExceptionGroup 与 BaseExceptionGroup](https://docs.python.org/3.12/library/exceptions.html#BaseExceptionGroup)、[except\* 的分组与限制](https://docs.python.org/3.12/reference/compound_stmts.html#except-star)；[assert 与优化模式](https://docs.python.org/3.12/reference/simple_stmts.html#the-assert-statement)；[print_exception](https://docs.python.org/3.12/library/traceback.html#traceback.print_exception)、[format_exception](https://docs.python.org/3.12/library/traceback.html#traceback.format_exception)、[extract_tb](https://docs.python.org/3.12/library/traceback.html#traceback.extract_tb)、[FrameSummary 的名称与位置](https://docs.python.org/3.12/library/traceback.html#traceback.FrameSummary)；[Pdb 的配置](https://docs.python.org/3.12/library/pdb.html#pdb.Pdb)、[runcall](https://docs.python.org/3.12/library/pdb.html#pdb.runcall)、[set_trace](https://docs.python.org/3.12/library/pdb.html#pdb.set_trace)、[p](https://docs.python.org/3.12/library/pdb.html#pdbcommand-p)、[next](https://docs.python.org/3.12/library/pdb.html#pdbcommand-next)、[step](https://docs.python.org/3.12/library/pdb.html#pdbcommand-step)、[list](https://docs.python.org/3.12/library/pdb.html#pdbcommand-list)、[where](https://docs.python.org/3.12/library/pdb.html#pdbcommand-where)、[up](https://docs.python.org/3.12/library/pdb.html#pdbcommand-up)、[down](https://docs.python.org/3.12/library/pdb.html#pdbcommand-down)、[break](https://docs.python.org/3.12/library/pdb.html#pdbcommand-break)、[continue](https://docs.python.org/3.12/library/pdb.html#pdbcommand-continue)、[quit](https://docs.python.org/3.12/library/pdb.html#pdbcommand-quit)；[breakpoint](https://docs.python.org/3.12/library/functions.html#breakpoint)、[sys.breakpointhook 与 PYTHONBREAKPOINT](https://docs.python.org/3.12/library/sys.html#sys.breakpointhook)、[StringIO 的内存流与关闭](https://docs.python.org/3.12/library/io.html#io.StringIO)、[cmd 的输入输出与 use_rawinput](https://docs.python.org/3.12/library/cmd.html#cmd.Cmd)。 |
| GitHub 官方源码 | [CPython 3.12.14：异常组要求非空序列](https://github.com/python/cpython/blob/v3.12.14/Objects/exceptions.c#L716-L733)；[ipykernel 7.3.0：替换 pdb.Pdb 与 set_trace](https://github.com/ipython/ipykernel/blob/v7.3.0/ipykernel/kernelapp.py#L730-L746)；[IPython 9.17.1：继承标准库 Pdb](https://github.com/ipython/ipython/blob/9.17.1/IPython/core/debugger.py#L104-L153)、[ipdb 提示符](https://github.com/ipython/ipython/blob/9.17.1/IPython/core/debugger.py#L162)、[Pdb 子类与配色](https://github.com/ipython/ipython/blob/9.17.1/IPython/core/debugger.py#L209-L320)、[InterruptiblePdb](https://github.com/ipython/ipython/blob/9.17.1/IPython/core/debugger.py#L1435-L1463)，仅说明本章宿主调试器与标准库的关系。 |